# MScFE 622 Stochastic Modeling — Group Work Project 2
## Regime-Based Rotation among TLT, GLD and SPY using VIX Regimes (Markov Chain / Gaussian HMM)

**Cara pakai (Google Colab):** *Runtime → Run all*. Notebook mengunduh data dari Yahoo Finance, menjalankan Langkah 1–5, menyimpan semua gambar ke folder `figs/`, dan di sel terakhir otomatis menyusun laporan Word `MScFE622_GWP2_Report.docx` (Technical + Non-Technical Report) berisi angka hasil run Anda.

Struktur: Step 1 (data) → Step 2 (model regime VIX) → Step 3 (pemilihan state) → Step 4 (aturan rotasi) → Step 5 (backtest) → Step 6 (laporan).

In [ ]:
%pip install -q hmmlearn yfinance python-docx

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from hmmlearn.hmm import GaussianHMM
warnings.filterwarnings("ignore")
plt.rcParams.update({'figure.dpi':110,'axes.grid':True,'grid.alpha':.3,'figure.figsize':(11,4.5),'axes.titlesize':12})
os.makedirs('figs', exist_ok=True)
SEED = 42; np.random.seed(SEED)
ASSETS = ['TLT','GLD','SPY']; TD = 252          # trading days / year
RF = 0.0                                          # risk-free rate (annual) used in Sharpe; assumption stated in report
COST_BPS = 5.0                                    # one-way transaction cost (bps) for the 'net of cost' variant
N_INIT = 20                                       # EM random restarts
PALETTE = {2:['#2ca02c','#d62728'], 3:['#2ca02c','#ff9900','#d62728']}
txt = {}                                          # narrative strings reused in the Word report

# Step 1 — Data Preparation and Exploration
**Question 1.1** Download daily adjusted close prices for TLT, GLD, SPY and VIX (maximum common sample).  
**Question 1.2** Compute daily log-returns for ETFs and the change (ΔVIX) / return of VIX.  
**Question 1.3** Align dates, drop missing values, and plot ETF returns and ΔVIX.

In [ ]:
# --- Q1.1 Download adjusted close prices (cached to CSV so the run is reproducible) ---
CACHE = 'gwp2_prices.csv'
def load_prices():
    if os.path.exists(CACHE):
        return pd.read_csv(CACHE, index_col=0, parse_dates=True)
    import yfinance as yf
    raw = yf.download(['TLT','GLD','SPY','^VIX'], start='1990-01-01', auto_adjust=False, progress=False)
    px = raw['Adj Close'].rename(columns={'^VIX':'VIX'})[['TLT','GLD','SPY','VIX']]
    px.to_csv(CACHE); return px

raw_px = load_prices()
first_valid = raw_px.apply(lambda s: s.first_valid_index())
print('First available date per series:'); print(first_valid.dt.date.to_string())
px = raw_px.dropna()                      # maximum COMMON sample (limited by the youngest series)
print(f"\nCommon sample: {px.index[0].date()} -> {px.index[-1].date()}  |  {len(px)} daily observations")
print('Missing values after alignment:', int(px.isna().sum().sum()))
px.tail()

In [ ]:
# --- Q1.2 Returns ---
ret  = np.log(px[ASSETS]).diff()                 # ETF daily log-returns
dvix = px['VIX'].diff().rename('dVIX')           # change in VIX (points) -> observation series for regime models
vixr = np.log(px['VIX']).diff().rename('VIX_logret')   # VIX log-return (for reference)

# --- Q1.3 Align, drop NaN ---
data = pd.concat([ret, dvix, vixr], axis=1).dropna()
px   = px.loc[data.index]
sret = np.expm1(ret.loc[data.index])             # simple returns (used for portfolio aggregation)
ret  = ret.loc[data.index]; dvix = data['dVIX']
N = len(data)
print(f'Clean sample: {data.index[0].date()} -> {data.index[-1].date()} | N = {N}')
desc = data.describe().T[['mean','std','min','max']]
desc['skew'] = data.skew(); desc['excess kurt'] = data.kurt()
desc['ann. mean'] = data.mean()*TD; desc['ann. vol'] = data.std()*np.sqrt(TD)
display(desc.round(5))
corr = data[ASSETS+['dVIX']].corr()
display(corr.round(3))

In [ ]:
# --- Q1.3 Plot 1: ETF returns over time ---
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, a, c in zip(axes, ASSETS, ['#1f77b4','#d4a017','#2ca02c']):
    ax.plot(ret.index, ret[a], lw=.6, color=c); ax.set_ylabel(f'{a} log-return'); ax.axhline(0, color='k', lw=.5)
axes[0].set_title('Daily log-returns of TLT, GLD and SPY'); axes[-1].set_xlabel('Date')
plt.tight_layout(); plt.savefig('figs/f1_returns.png', dpi=150); plt.show()

# --- Q1.3 Plot 2: change in VIX ---
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(px.index, px['VIX'], lw=.8, color='#444'); axes[0].set_ylabel('VIX level')
axes[0].set_title('VIX level and daily change (ΔVIX)')
axes[1].plot(dvix.index, dvix, lw=.6, color='#8c1c13'); axes[1].axhline(0, color='k', lw=.5)
axes[1].set_ylabel('ΔVIX (index points)'); axes[1].set_xlabel('Date')
plt.tight_layout(); plt.savefig('figs/f2_dvix.png', dpi=150); plt.show()

txt['step1'] = (f"The common sample runs from {data.index[0].date()} to {data.index[-1].date()} ({N:,} daily observations); it starts when the youngest series "
    f"({first_valid.idxmax()}, first price {first_valid.max().date()}) becomes available. No missing values remain after alignment. "
    f"Daily volatility is {ret['SPY'].std()*np.sqrt(TD):.1%} (SPY), {ret['GLD'].std()*np.sqrt(TD):.1%} (GLD) and {ret['TLT'].std()*np.sqrt(TD):.1%} (TLT) annualised. "
    f"ΔVIX has mean {dvix.mean():.3f}, standard deviation {dvix.std():.2f} points, skewness {dvix.skew():.2f} and excess kurtosis {dvix.kurt():.1f}: strong fat tails and volatility clustering, "
    f"so a single Gaussian is a poor description and a regime (mixture) structure is natural. The correlation of ΔVIX with SPY returns is {corr.loc['SPY','dVIX']:.2f}, "
    f"with TLT {corr.loc['TLT','dVIX']:.2f} and with GLD {corr.loc['GLD','dVIX']:.2f}.")
print(txt['step1'])

# Step 2 — Modeling VIX Regimes
Kedua opsi dikerjakan: **(a)** Discrete Markov Chain (state dari kuantil ΔVIX) dan **(b)** Gaussian HMM 2- & 3-state yang diestimasi dengan algoritma EM (Baum–Welch), dengan 20 *random restarts* dan pemilihan log-likelihood tertinggi. State HMM diurutkan menurut simpangan baku emisi (state 0 = paling tenang).

In [ ]:
# ---------- (a) Discrete Markov chain ----------
def mc_fit(x, k):
    qs = np.quantile(x, np.linspace(0, 1, k+1)[1:-1])
    s = pd.Series(np.digitize(x.values, qs), index=x.index)
    Cn = np.zeros((k, k))
    for a, b in zip(s.values[:-1], s.values[1:]): Cn[a, b] += 1
    P = Cn / Cn.sum(axis=1, keepdims=True)
    w, v = np.linalg.eig(P.T); pi = np.real(v[:, np.argmin(abs(w-1))]); pi = pi/pi.sum()
    ll = float((Cn[Cn>0]*np.log(P[Cn>0])).sum())
    return dict(kind='MC', k=k, states=s, C=Cn, P=P, pi=pi, ll=ll, npar=k*(k-1), nobs=len(s)-1, edges=qs)

MC = {k: mc_fit(dvix, k) for k in (2, 3)}
for k in (2, 3):
    print(f'--- Markov chain, {k} states (quantile cut-offs of ΔVIX: {np.round(MC[k]["edges"],3)}) ---')
    print('Transition matrix P:'); display(pd.DataFrame(MC[k]['P'], index=[f'from {i}' for i in range(k)], columns=[f'to {i}' for i in range(k)]).round(4))
    print('Stationary distribution:', np.round(MC[k]['pi'], 4), '| empirical freq:', np.round(MC[k]['states'].value_counts(normalize=True).sort_index().values, 4))

In [ ]:
# ---------- (b) Gaussian HMM via EM ----------
def hmm_fit(x, k, n_init=N_INIT, seed=SEED):
    X = x.values.reshape(-1, 1); best, best_ll = None, -np.inf
    for i in range(n_init):
        m = GaussianHMM(n_components=k, covariance_type='full', n_iter=2000, tol=1e-7, random_state=seed+i)
        try:
            m.fit(X); ll = m.score(X)
        except Exception:
            continue
        if np.isfinite(ll) and ll > best_ll: best, best_ll = m, ll
    return best

def hmm_pack(m, x, k):
    X = x.values.reshape(-1, 1)
    sd = np.sqrt(m.covars_.reshape(k)); order = np.argsort(sd); inv = np.argsort(order)
    mu = m.means_.reshape(k)[order]; sd = sd[order]
    A = m.transmat_[np.ix_(order, order)]; pi0 = m.startprob_[order]
    states = pd.Series(inv[m.predict(X)], index=x.index)                       # Viterbi path
    prob = pd.DataFrame(m.predict_proba(X)[:, order], index=x.index, columns=range(k))   # smoothed probabilities
    w, v = np.linalg.eig(A.T); st = np.real(v[:, np.argmin(abs(w-1))]); st = st/st.sum()
    return dict(kind='HMM', k=k, mu=mu, sd=sd, A=A, pi0=pi0, pi=st, states=states, prob=prob,
                ll=float(m.score(X)), npar=k*k+2*k-1, nobs=len(x), model=m)

HMM = {k: hmm_pack(hmm_fit(dvix, k), dvix, k) for k in (2, 3)}
vix_lvl = px['VIX']
for k in (2, 3):
    r = HMM[k]
    print(f'=== Gaussian HMM, {k} states: log-likelihood = {r["ll"]:.2f} ===')
    par = pd.DataFrame({'mean ΔVIX': r['mu'], 'std ΔVIX': r['sd'], 'persistence a_ii': np.diag(r['A']),
                        'expected duration (days)': 1/(1-np.diag(r['A'])), 'stationary prob': r['pi'],
                        'share of days (Viterbi)': r['states'].value_counts(normalize=True).reindex(range(k)).fillna(0).values,
                        'mean VIX level': vix_lvl.groupby(r['states']).mean().reindex(range(k)).values},
                       index=[f'state {i}' for i in range(k)])
    display(par.round(4))
    print('Transition matrix A:'); display(pd.DataFrame(r['A'], index=[f'from {i}' for i in range(k)], columns=[f'to {i}' for i in range(k)]).round(4))
    print('Most-likely state sequence (first 10):', r['states'].values[:10], '| smoothed probabilities (first 3 rows):'); display(r['prob'].head(3).round(3))

In [ ]:
# ---------- Plot: VIX with colour-coded states ----------
def plot_vix_states(states, k, title, fname, labels):
    fig, ax = plt.subplots(figsize=(12, 4.8))
    ax.plot(vix_lvl.index, vix_lvl.values, color='lightgrey', lw=.8, zorder=1)
    for s in range(k):
        m = (states.values == s)
        ax.scatter(vix_lvl.index[m], vix_lvl.values[m], s=5, color=PALETTE[k][s], label=labels[s], zorder=2)
    ax.set_title(title); ax.set_xlabel('Date'); ax.set_ylabel('VIX level'); ax.legend(markerscale=3, loc='upper right')
    plt.tight_layout(); plt.savefig(fname, dpi=150); plt.show()

LAB = {2:['State 0: calm','State 1: turbulent'], 3:['State 0: calm','State 1: moderate','State 2: turbulent']}
LAB_MC = {2:['ΔVIX low (VIX falling)','ΔVIX high (VIX rising)'], 3:['ΔVIX low (VIX falling)','ΔVIX mid','ΔVIX high (VIX rising)']}
plot_vix_states(HMM[2]['states'], 2, 'VIX coloured by Gaussian-HMM state (2 states)', 'figs/f3_states_hmm2.png', LAB[2])
plot_vix_states(HMM[3]['states'], 3, 'VIX coloured by Gaussian-HMM state (3 states)', 'figs/f3_states_hmm3.png', LAB[3])
plot_vix_states(MC[3]['states'], 3, 'VIX coloured by Markov-chain state (3 quantile states of ΔVIX)', 'figs/f3_states_mc3.png', LAB_MC[3])

# Step 3 — State Selection and Interpretation
**Question 3.1** Bandingkan model dengan log-likelihood, AIC, BIC. **Question 3.2** Pilih model dan jelaskan. **Question 3.3** Mean & std return ETF per state + bar chart.

*Catatan:* log-likelihood Markov chain adalah likelihood *diskret* (bersyarat pada state awal), sehingga **tidak sebanding secara langsung** dengan likelihood densitas Gaussian HMM. Pemilihan formal dilakukan di antara model HMM; MC dilaporkan sebagai pembanding.

In [ ]:
rows = {}
for kind, D in (('MC', MC), ('HMM', HMM)):
    for k, r in D.items():
        T = r['P'] if kind == 'MC' else r['A']
        occ = r['states'].value_counts(normalize=True).reindex(range(k)).fillna(0)
        rows[f'{kind}-{k}'] = dict(Method=kind, States=k, Params=r['npar'], N=r['nobs'], LogLik=r['ll'],
                                   AIC=2*r['npar']-2*r['ll'], BIC=r['npar']*np.log(r['nobs'])-2*r['ll'],
                                   MinOccupancy=occ.min(), MinPersistence=np.diag(T).min(),
                                   Comparable='yes (densities)' if kind=='HMM' else 'no (discrete likelihood)')
tab = pd.DataFrame(rows).T
for c in ['Params','N','States']: tab[c] = tab[c].astype(int)
for c in ['LogLik','AIC','BIC','MinOccupancy','MinPersistence']: tab[c] = tab[c].astype(float)
display(tab.round(3))

In [ ]:
# --- Q3.2 Model selection: lowest BIC among HMMs that are also stable/interpretable ---
CHOSEN = None        # <- set to 'HMM-2' or 'HMM-3' (or 'MC-3') to override the automatic choice
cand = tab[tab.Method == 'HMM']
stable = cand[(cand.MinOccupancy >= .05) & (cand.MinPersistence >= .5)]
pool = stable if len(stable) else cand
CH = CHOSEN or pool.BIC.idxmin()
chosen = (HMM if CH.startswith('HMM') else MC)[int(CH.split('-')[1])]
K = chosen['k']; states = chosen['states']
aic_best, bic_best = cand.AIC.idxmin(), cand.BIC.idxmin()
other = [i for i in cand.index if i != CH][0]
txt['step3_choice'] = (f"The preferred specification is the {K}-state Gaussian HMM ({CH}). Among the HMMs, BIC is lowest for {bic_best} "
    f"(BIC {cand.loc[bic_best,'BIC']:.1f} vs {cand.loc[other if bic_best==CH else CH,'BIC']:.1f}) and AIC is lowest for {aic_best}. " +
    (f"Both candidates pass the stability screen (every state occupies at least {cand.MinOccupancy.min():.1%} of the sample and persistence a_ii of at least {cand.MinPersistence.min():.2f}). " if len(stable)==len(cand) else f"Only {', '.join(stable.index) or 'none'} of the HMMs pass the stability screen (min. state share 5%, min. persistence 0.5), so the choice is restricted accordingly. ") +
    f"Average persistence of the quantile-based chains is {np.mean(np.diag(MC[3]['P'])):.2f} (3 states) versus {np.mean(np.diag(HMM[3]['A'])):.2f} for the 3-state HMM. "
    f"The selected model has persistence a_ii = {', '.join(f'{x:.3f}' for x in np.diag(chosen['A']))}, i.e. expected durations of "
    f"{', '.join(f'{x:.1f}' for x in 1/(1-np.diag(chosen['A'])))} trading days, and stationary probabilities {', '.join(f'{x:.2f}' for x in chosen['pi'])}. "
    f"The Markov chains built from ΔVIX quantiles are reported for comparison only, because their discrete likelihood is not comparable with the HMM density likelihood.")
print('Chosen model:', CH); print(txt['step3_choice'])

In [ ]:
# --- Q3.3 State-conditional ETF returns (same-day states, Viterbi path of chosen model) ---
g = ret.groupby(states)
cond_mean, cond_std = g.mean().reindex(range(K)), g.std().reindex(range(K))
cond_n = states.value_counts().reindex(range(K)).fillna(0).astype(int)
cond = pd.concat({'mean (daily)': cond_mean, 'std (daily)': cond_std, 'mean (ann.)': cond_mean*TD, 'std (ann.)': cond_std*np.sqrt(TD)}, axis=1)
cond['days'] = cond_n; display(cond.round(5))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
w = .25; xs = np.arange(K)
for j, a in enumerate(ASSETS):
    axes[0].bar(xs + (j-1)*w, cond_mean[a]*1e4, w, label=a)
    axes[1].bar(xs + (j-1)*w, cond_std[a]*1e4, w, label=a)
for ax, t, yl in zip(axes, ['Mean daily log-return by regime', 'Std. dev. of daily log-return by regime'], ['basis points per day', 'basis points per day']):
    ax.set_xticks(xs); ax.set_xticklabels([LAB[K][s] if CH.startswith('HMM') else LAB_MC[K][s] for s in xs], fontsize=8)
    ax.set_title(t); ax.set_ylabel(yl); ax.set_xlabel('Regime'); ax.legend(); ax.axhline(0, color='k', lw=.6)
plt.tight_layout(); plt.savefig('figs/f4_state_returns.png', dpi=150); plt.show()

sent = []
for s in range(K):
    top = cond_mean.loc[s].idxmax(); lo = cond_mean.loc[s].idxmin()
    sent.append(f"In state {s} ({cond_n[s]:,} days) the highest mean return is {top} ({cond_mean.loc[s,top]*1e4:.1f} bp/day) and the lowest is {lo} ({cond_mean.loc[s,lo]*1e4:.1f} bp/day); "
                f"SPY volatility is {cond_std.loc[s,'SPY']*np.sqrt(TD):.1%} annualised.")
txt['step3_cond'] = ' '.join(sent)
print(txt['step3_cond'])

# Step 4 — Designing the Rotation Strategy
**Aturan:** untuk setiap state, alokasikan 100% ke ETF dengan *mean return historis tertinggi* pada state tersebut (variasi opsional: 60/40 untuk dua ETF teratas). **Eksekusi dengan lag 1 hari**: bobot hari *t* ditentukan oleh state pada penutupan hari *t−1*.

In [ ]:
def mapping_table(st, r, k, train=None, top2=False):
    rr = r if train is None else r.loc[train]; ss = st if train is None else st.loc[train]
    mu = rr.groupby(ss).mean().reindex(range(k)); mu = mu.fillna(rr.mean())
    W = pd.DataFrame(0.0, index=range(k), columns=ASSETS)
    for s in range(k):
        o = mu.loc[s].sort_values(ascending=False).index
        if top2: W.loc[s, o[0]] = .6; W.loc[s, o[1]] = .4
        else:    W.loc[s, o[0]] = 1.
    return W, mu

def weights_from_states(st, W, lag=1):
    w = pd.DataFrame(W.values[st.values.astype(int)], index=st.index, columns=ASSETS)
    return w.shift(lag)

W1, mu_state = mapping_table(states, ret, K)               # 100% to the best asset
W2, _        = mapping_table(states, ret, K, top2=True)    # 60/40 variant
names = (LAB if CH.startswith('HMM') else LAB_MC)[K]
map_tab = pd.DataFrame({'Regime': names,
    'Best ETF (mean)': [mu_state.loc[s].idxmax() for s in range(K)],
    'Mean (bp/day)': [round(mu_state.loc[s].max()*1e4, 2) for s in range(K)],
    'Second ETF': [mu_state.loc[s].sort_values(ascending=False).index[1] for s in range(K)],
    'Weights 100% rule': [', '.join(f'{a} {W1.loc[s,a]:.0%}' for a in ASSETS if W1.loc[s,a] > 0) for s in range(K)],
    'Weights 60/40 rule': [', '.join(f'{a} {W2.loc[s,a]:.0%}' for a in ASSETS if W2.loc[s,a] > 0) for s in range(K)]}, index=[f'state {s}' for s in range(K)])
display(map_tab)
txt['step4'] = ("The decision rule maps each regime to the asset with the highest historical mean return in that regime: " +
    '; '.join(f"{names[s]} → {map_tab.iloc[s]['Best ETF (mean)']}" for s in range(K)) +
    ". The signal is observed at the close of day t−1 and traded for day t (1-day execution lag), which removes same-day look-ahead. "
    "The rule ranks assets on mean return only and ignores risk and estimation error; it is therefore transparent but simple, and the 60/40 variant is shown as a diversifying alternative.")
print(txt['step4'])

# Step 5 — Backtesting and Evaluation
Metrik: Cumulative Return, Annualized Return (geometrik), Volatility (annualized), Sharpe Ratio (rf = 0), Max Drawdown. Benchmark: **Equal-weight 1/3 rebalance bulanan** dan **Buy-and-Hold SPY**. Semua seri dievaluasi pada jendela tanggal yang sama.

In [ ]:
def metrics(r, rf=RF):
    eq = (1+r).cumprod(); n = len(r)
    ann = eq.iloc[-1]**(TD/n) - 1; vol = r.std()*np.sqrt(TD)
    return pd.Series({'Cumulative Return': eq.iloc[-1]-1, 'Annualized Return': ann, 'Volatility': vol,
                      'Sharpe Ratio': (r.mean()*TD - rf)/vol, 'Max Drawdown': (eq/eq.cummax()-1).min()})

def backtest(w, sr, cost_bps=0.0):
    w = w.dropna(); r = (w*sr.loc[w.index]).sum(axis=1)
    turn = w.diff().abs().sum(axis=1).fillna(0)
    return r - turn*cost_bps/1e4

def eq_weight_monthly(sr):
    out = pd.Series(index=sr.index, dtype=float)
    for _, g in sr.groupby([sr.index.year, sr.index.month]):
        port = (1+g).cumprod().mean(axis=1)               # equal weights reset at start of each month, drift inside month
        out.loc[g.index] = port.values/np.r_[1.0, port.values[:-1]] - 1
    return out

w1 = weights_from_states(states, W1); w2 = weights_from_states(states, W2)
idx = w1.dropna().index
R = pd.DataFrame({
    'Regime rotation (100%)': backtest(w1, sret),
    f'Regime rotation (100%, net {COST_BPS:.0f} bp)': backtest(w1, sret, COST_BPS),
    'Regime rotation (60/40)': backtest(w2, sret),
    'Equal-weight (monthly rebal.)': eq_weight_monthly(sret.loc[idx]),
    'Buy-and-hold SPY': sret.loc[idx, 'SPY']}).loc[idx]
perf = R.apply(metrics).T
display(perf.style.format({'Cumulative Return':'{:.1%}','Annualized Return':'{:.2%}','Volatility':'{:.2%}','Sharpe Ratio':'{:.2f}','Max Drawdown':'{:.1%}'}))
turn1 = w1.dropna().diff().abs().sum(axis=1).fillna(0)
print(f'Average one-way turnover of the 100% rule: {turn1.mean()*TD:.1f} (sum |Δw|) per year; share of days with a trade: {(turn1>0).mean():.1%}')

In [ ]:
eq = (1+R).cumprod()
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={'height_ratios':[2, 1]})
sty = {'Regime rotation (100%)':('#d62728',2.0), f'Regime rotation (100%, net {COST_BPS:.0f} bp)':('#d62728',1.0), 'Regime rotation (60/40)':('#ff7f0e',1.4),
       'Equal-weight (monthly rebal.)':('#1f77b4',1.4), 'Buy-and-hold SPY':('#2ca02c',1.4)}
for c in eq.columns:
    axes[0].plot(eq.index, eq[c], label=c, color=sty[c][0], lw=sty[c][1], ls='--' if 'net' in c else '-')
    axes[1].plot(eq.index, eq[c]/eq[c].cummax()-1, color=sty[c][0], lw=.9)
axes[0].set_yscale('log'); axes[0].set_ylabel('Growth of 1 (log scale)'); axes[0].set_title('Cumulative performance (1-day execution lag)'); axes[0].legend(fontsize=8)
axes[1].set_ylabel('Drawdown'); axes[1].set_xlabel('Date')
plt.tight_layout(); plt.savefig('figs/f5_cumulative.png', dpi=150); plt.show()

S, B_EW, B_SPY = perf.loc['Regime rotation (100%)'], perf.loc['Equal-weight (monthly rebal.)'], perf.loc['Buy-and-hold SPY']
def verdict(a, b, nm): return f"{'higher' if a['Sharpe Ratio']>b['Sharpe Ratio'] else 'lower'} than {nm} ({a['Sharpe Ratio']:.2f} vs {b['Sharpe Ratio']:.2f})"
txt['step5'] = (f"Over {idx[0].date()} – {idx[-1].date()} the regime rotation (100% rule) returned {S['Annualized Return']:.2%} per year with volatility {S['Volatility']:.2%}, Sharpe {S['Sharpe Ratio']:.2f} and maximum drawdown {S['Max Drawdown']:.1%}. "
    f"Its Sharpe ratio is {verdict(S, B_SPY, 'buy-and-hold SPY')} and {verdict(S, B_EW, 'the equal-weight portfolio')}. "
    f"The maximum drawdown is {S['Max Drawdown']:.1%} versus {B_SPY['Max Drawdown']:.1%} (SPY) and {B_EW['Max Drawdown']:.1%} (equal-weight). "
    f"With a {COST_BPS:.0f} bp one-way cost the Sharpe ratio becomes {perf.loc[f'Regime rotation (100%, net {COST_BPS:.0f} bp)','Sharpe Ratio']:.2f}. The 60/40 variant has Sharpe {perf.loc['Regime rotation (60/40)','Sharpe Ratio']:.2f} and drawdown {perf.loc['Regime rotation (60/40)','Max Drawdown']:.1%}.")
print(txt['step5'])

In [ ]:
# --- Q5.5 Sensitivity: number of states, method, sample window, and a causal out-of-sample check ---
def forward_filter(res, x):
    mu, sd, A = res['mu'], res['sd'], res['A']; k = len(mu); b = norm.pdf(x[:, None], mu, sd)
    al = np.zeros((len(x), k)); a = res['pi']*b[0]; a = a/a.sum(); al[0] = a
    for t in range(1, len(x)):
        a = (al[t-1] @ A)*b[t]; s = a.sum(); a = a/s if s > 0 else np.ones(k)/k; al[t] = a
    return al

def row(label, st, k, r_, sr_, train=None, window=None):
    W, _ = mapping_table(st, r_, k, train=train)
    w = weights_from_states(st, W).dropna()
    if window is not None: w = w.loc[window]
    s = backtest(w, sr_); sp = sr_.loc[s.index, 'SPY']; ew = eq_weight_monthly(sr_.loc[s.index])
    ms, mp, me = metrics(s), metrics(sp), metrics(ew)
    return dict(Scenario=label, Period=f'{s.index[0].date()} – {s.index[-1].date()}', **{'Strategy Ann.Ret': ms['Annualized Return'], 'Strategy Sharpe': ms['Sharpe Ratio'], 'Strategy MaxDD': ms['Max Drawdown'],
            'SPY Sharpe': mp['Sharpe Ratio'], 'EW Sharpe': me['Sharpe Ratio']})

sens = [row('HMM 2 states (in-sample)', HMM[2]['states'], 2, ret, sret),
        row('HMM 3 states (in-sample)', HMM[3]['states'], 3, ret, sret),
        row('Markov chain 3 states (in-sample)', MC[3]['states'], 3, ret, sret)]
# sample window: re-estimate on a shorter window (post-2010), same method as chosen model
sub = data.loc['2010-01-01':]
if len(sub) > 750 and CH.startswith('HMM'):
    rs = hmm_pack(hmm_fit(sub['dVIX'], K), sub['dVIX'], K)
    sens.append(row(f'{CH} re-estimated on 2010+ window', rs['states'], K, ret.loc[sub.index], sret.loc[sub.index]))
# causal out-of-sample: estimate on first half, filter forward (no future data) on second half
half = N//2; tr = data.index[:half]
if CH.startswith('HMM'):
    rtr = hmm_pack(hmm_fit(dvix.iloc[:half], K), dvix.iloc[:half], K)
    alpha = forward_filter(rtr, dvix.values)
    st_oos = pd.Series(alpha.argmax(axis=1), index=dvix.index)
    sens.append(row(f'{CH} out-of-sample (fit first half, test second half)', st_oos, K, ret, sret, train=tr, window=data.index[half:]))
sens = pd.DataFrame(sens).set_index('Scenario')
display(sens.style.format({'Strategy Ann.Ret':'{:.2%}','Strategy Sharpe':'{:.2f}','Strategy MaxDD':'{:.1%}','SPY Sharpe':'{:.2f}','EW Sharpe':'{:.2f}'}))

oos_txt = ''
if CH.startswith('HMM'):
    o = sens.iloc[-1]
    oos_txt = (f" In the causal out-of-sample test (parameters and the asset mapping estimated on the first half only) the strategy's Sharpe ratio is {o['Strategy Sharpe']:.2f} "
               f"against {o['SPY Sharpe']:.2f} for SPY and {o['EW Sharpe']:.2f} for the equal-weight portfolio over {o['Period']}.")
txt['step5_sens'] = (f"Changing the number of states or method moves the Sharpe ratio between {sens['Strategy Sharpe'].min():.2f} and {sens['Strategy Sharpe'].max():.2f} across the scenarios, "
    f"so the result depends materially on the regime definition and the sample window.{oos_txt} "
    "The full-sample regime path and the mapping use information from the whole sample, so the in-sample results are optimistic; the out-of-sample row is the more honest estimate of tradable performance.")
print(txt['step5_sens'])

# Step 6 — Laporan Word (Technical + Non-Technical)
Sel berikut menyusun `MScFE622_GWP2_Report.docx` dari hasil di atas (angka, tabel, dan gambar terisi otomatis; tidak memuat kode, sesuai ketentuan rubrik). **Isi nama grup/anggota di halaman pertama** (variabel `COVER`). Tinjau dan sesuaikan narasi sebelum submit — interpretasi otomatis ini adalah draf berbasis angka Anda.

In [ ]:
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

COVER = {'Course':'MScFE 622 Stochastic Modeling', 'Assignment':'Group Work Project 2 — Regime-Based Rotation (TLT, GLD, SPY) on VIX Regimes',
         'Group number':'[isi nomor grup]', 'Submitted by':'Ir. Agus Budianto Ulil Absyor, S.T., M.M., M.LSCM., M.AIB. (Student ID 1030907)',
         'Group members':'[isi nama & ID anggota grup]', 'Date':str(pd.Timestamp.today().date())}
REFS = [
 "Akaike, Hirotugu. \"A New Look at the Statistical Model Identification.\" IEEE Transactions on Automatic Control, vol. 19, no. 6, 1974, pp. 716–723.",
 "Ang, Andrew, and Geert Bekaert. \"International Asset Allocation with Regime Shifts.\" The Review of Financial Studies, vol. 15, no. 4, 2002, pp. 1137–1187.",
 "Dempster, A. P., N. M. Laird, and D. B. Rubin. \"Maximum Likelihood from Incomplete Data via the EM Algorithm.\" Journal of the Royal Statistical Society: Series B, vol. 39, no. 1, 1977, pp. 1–38.",
 "Hamilton, James D. \"A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle.\" Econometrica, vol. 57, no. 2, 1989, pp. 357–384.",
 "Rabiner, Lawrence R. \"A Tutorial on Hidden Markov Models and Selected Applications in Speech Recognition.\" Proceedings of the IEEE, vol. 77, no. 2, 1989, pp. 257–286.",
 "Schwarz, Gideon. \"Estimating the Dimension of a Model.\" The Annals of Statistics, vol. 6, no. 2, 1978, pp. 461–464.",
 "Sharpe, William F. \"The Sharpe Ratio.\" The Journal of Portfolio Management, vol. 21, no. 1, 1994, pp. 49–58.",
 "Whaley, Robert E. \"The Investor Fear Gauge.\" The Journal of Portfolio Management, vol. 26, no. 3, 2000, pp. 12–17.",
 "Yahoo Finance. Historical Prices for TLT, GLD, SPY and ^VIX. Yahoo, finance.yahoo.com. Accessed " + str(pd.Timestamp.today().date()) + "."]

doc = Document()
st_ = doc.styles['Normal']; st_.font.name = 'Calibri'; st_.font.size = Pt(11)
for s in doc.sections: s.left_margin = s.right_margin = Inches(1); s.top_margin = s.bottom_margin = Inches(.9)

def shade(cell, color):
    tcPr = cell._tc.get_or_add_tcPr(); sh = OxmlElement('w:shd'); sh.set(qn('w:val'), 'clear'); sh.set(qn('w:color'), 'auto'); sh.set(qn('w:fill'), color); tcPr.append(sh)
def H(t, l=1): return doc.add_heading(t, level=l)
def P(t, bold=False, italic=False):
    p = doc.add_paragraph(); r = p.add_run(t); r.bold = bold; r.italic = italic; p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY; return p
def BUL(items):
    for i in items: doc.add_paragraph(i, style='List Bullet')
def FIG(path, cap, w=6.3):
    doc.add_picture(path, width=Inches(w)); doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
    c = doc.add_paragraph(cap); c.alignment = WD_ALIGN_PARAGRAPH.CENTER; c.runs[0].italic = True; c.runs[0].font.size = Pt(9)
def TBL(df, fmt=None, caption=None, index=True, size=8.5):
    if caption: 
        c = doc.add_paragraph(caption); c.runs[0].bold = True; c.runs[0].font.size = Pt(9.5)
    d = df.reset_index() if index else df.copy()
    t = doc.add_table(rows=1, cols=len(d.columns)); t.style = 'Table Grid'
    for j, col in enumerate(d.columns):
        cell = t.rows[0].cells[j]; cell.text = ''; r = cell.paragraphs[0].add_run(str(col)); r.bold = True; r.font.size = Pt(size); r.font.color.rgb = RGBColor(255,255,255); shade(cell, '3B3355')
    for _, rw in d.iterrows():
        cells = t.add_row().cells
        for j, col in enumerate(d.columns):
            v = rw[col]; f = (fmt or {}).get(col)
            s = (f.format(v) if f and isinstance(v, (int, float, np.floating, np.integer)) else (f'{v:.4g}' if isinstance(v, (float, np.floating)) else str(v)))
            cells[j].text = ''; r = cells[j].paragraphs[0].add_run(s); r.font.size = Pt(size)
    doc.add_paragraph()

# ---- cover page ----
t = doc.add_heading('MScFE 622 Stochastic Modeling — Group Work Project 2', 0)
for k_, v_ in COVER.items():
    p = doc.add_paragraph(); r = p.add_run(f'{k_}: '); r.bold = True; p.add_run(v_)
doc.add_page_break()

# ---- PART A: technical ----
H('Part A — Technical Report', 1)
H('Step 1. Data preparation and exploration', 2)
P(txt['step1']); FIG('figs/f1_returns.png', 'Figure 1. Daily log-returns of TLT, GLD and SPY.'); FIG('figs/f2_dvix.png', 'Figure 2. VIX level and daily change ΔVIX (index points).')
TBL(desc[['mean','std','skew','excess kurt','ann. mean','ann. vol']].round(5), caption='Table 1. Descriptive statistics (daily unless stated).')

H('Step 2. Modeling VIX regimes', 2)
P("Two model families were estimated on ΔVIX. (i) A discrete Markov chain with states defined by sample quantiles of ΔVIX (2 and 3 states); transition probabilities are the empirical conditional frequencies (maximum-likelihood estimates) and the stationary distribution solves πP = π. "
  "(ii) Gaussian hidden Markov models with 2 and 3 states, estimated by the Expectation–Maximisation (Baum–Welch) algorithm with 20 random initialisations, retaining the highest log-likelihood; the most-likely path is obtained by the Viterbi algorithm and smoothed probabilities by the forward–backward algorithm. States are ordered by emission standard deviation (state 0 = calmest).")
for k in (2, 3):
    r = HMM[k]
    par = pd.DataFrame({'Mean ΔVIX': r['mu'], 'Std ΔVIX': r['sd'], 'a_ii': np.diag(r['A']), 'Exp. duration (days)': 1/(1-np.diag(r['A'])), 'Stationary prob.': r['pi'],
        'Share of days': r['states'].value_counts(normalize=True).reindex(range(k)).fillna(0).values, 'Mean VIX level': vix_lvl.groupby(r['states']).mean().reindex(range(k)).values}, index=[f'State {i}' for i in range(k)])
    TBL(par.round(4), caption=f'Table 2.{k-1}. Fitted parameters, Gaussian HMM with {k} states (log-likelihood {r["ll"]:.1f}).')
    TBL(pd.DataFrame(r['A'], index=[f'from {i}' for i in range(k)], columns=[f'to {i}' for i in range(k)]).round(4), caption=f'Transition matrix A, HMM with {k} states.')
TBL(pd.DataFrame(MC[3]['P'], index=[f'from {i}' for i in range(3)], columns=[f'to {i}' for i in range(3)]).round(4), caption='Table 2.3. Transition matrix, Markov chain with 3 ΔVIX-quantile states '
    f'(stationary distribution {", ".join(f"{x:.3f}" for x in MC[3]["pi"])}).')
FIG('figs/f3_states_hmm2.png', 'Figure 3. VIX coloured by 2-state HMM regime.'); FIG('figs/f3_states_hmm3.png', 'Figure 4. VIX coloured by 3-state HMM regime.')
P("Interpretation. " + " ".join(f"In the {k}-state HMM, state {i} has ΔVIX standard deviation {HMM[k]['sd'][i]:.2f} points, mean VIX level {vix_lvl.groupby(HMM[k]['states']).mean().get(i, float('nan')):.1f} and expected duration {1/(1-HMM[k]['A'][i,i]):.1f} days." for k in (2,3) for i in range(k)) +
  " States are therefore distinguished mainly by the dispersion of VIX changes and, consequently, by the VIX level. Persistent diagonal entries are what make a regime-based rule implementable, whereas quantile-defined chains classify each day by the size of a single day's change.")

H('Step 3. State selection and interpretation', 2)
tab_r = tab.copy(); tab_r.index.name = 'Model'
tab_r = tab_r.assign(**{c: tab_r[c].map('{:,.1f}'.format) for c in ['LogLik','AIC','BIC']}, **{c: tab_r[c].map('{:.3f}'.format) for c in ['MinOccupancy','MinPersistence']})
TBL(tab_r[['Method','States','Params','LogLik','AIC','BIC','MinOccupancy','MinPersistence','Comparable']], caption='Table 3. Model comparison (log-likelihood, AIC, BIC).', size=7.5)
P(txt['step3_choice'])
cd = pd.concat({'Mean (bp/day)': cond_mean*1e4, 'Std (bp/day)': cond_std*1e4}, axis=1); cd.columns = [f'{a} {b}' for a, b in cd.columns]; cd['Days'] = cond_n.values; cd.index = [f'State {s}' for s in range(K)]
TBL(cd.round(2), caption='Table 4. State-conditional ETF returns (basis points per day).')
FIG('figs/f4_state_returns.png', 'Figure 5. Mean and standard deviation of daily ETF log-returns by regime.')
P(txt['step3_cond'])

H('Step 4. Rotation strategy', 2)
TBL(map_tab, caption='Table 5. Final state → allocation mapping.', size=8)
P(txt['step4'])

H('Step 5. Backtesting and evaluation', 2)
pf = perf.copy(); pf.index.name = 'Strategy'
TBL(pf.assign(**{c: pf[c].map('{:.1%}'.format) for c in ['Cumulative Return','Max Drawdown']}, **{c: pf[c].map('{:.2%}'.format) for c in ['Annualized Return','Volatility']}, **{'Sharpe Ratio': pf['Sharpe Ratio'].map('{:.2f}'.format)}),
    caption='Table 6. Performance summary (1-day execution lag; risk-free rate 0%).', size=8)
FIG('figs/f5_cumulative.png', 'Figure 6. Cumulative performance (log scale) and drawdowns.')
P(txt['step5'])
sn = sens.copy(); sn.index.name = 'Scenario'
TBL(sn.assign(**{c: sn[c].map('{:.2f}'.format) for c in ['Strategy Sharpe','SPY Sharpe','EW Sharpe']}, **{'Strategy Ann.Ret': sn['Strategy Ann.Ret'].map('{:.2%}'.format), 'Strategy MaxDD': sn['Strategy MaxDD'].map('{:.1%}'.format)}),
    caption='Table 7. Sensitivity to number of states, method and sample window.', size=7.5)
P(txt['step5_sens'])
P("Limitations. The mapping ranks assets on in-sample mean returns, which are noisy estimates; regime labels are inferred with hindsight in the in-sample runs; the Sharpe ratio assumes a zero risk-free rate; transaction costs are approximated by a flat per-turnover charge; and past regime behaviour need not persist.", italic=True)

# ---- PART B: non-technical ----
doc.add_page_break(); H('Part B — Non-Technical Report', 1)
H('1. What we found', 2)
P(f"We studied whether the market's \"fear gauge\" (VIX) can tell us when to hold US Treasury bonds (TLT), gold (GLD) or US stocks (SPY). Using {N:,} trading days of data, we classified each day as a calm or a stressed market period and looked at what each asset did in each period. "
  f"{txt['step3_cond']}")
P(f"Following a simple rule (hold whichever asset has done best in the current type of market, acting one day after the signal), a $100 investment would have grown to ${100*eq['Regime rotation (100%)'].iloc[-1]:.0f}, versus ${100*eq['Buy-and-hold SPY'].iloc[-1]:.0f} for holding SPY and ${100*eq['Equal-weight (monthly rebal.)'].iloc[-1]:.0f} for an equal-weight mix. "
  f"The rotation delivered a risk-adjusted return (Sharpe) of {S['Sharpe Ratio']:.2f} compared with {B_SPY['Sharpe Ratio']:.2f} for SPY and {B_EW['Sharpe Ratio']:.2f} for equal-weight, and its worst peak-to-trough loss was {S['Max Drawdown']:.1%} versus {B_SPY['Max Drawdown']:.1%} for SPY.")
H('2. Recommended course of action', 2)
imp = S['Sharpe Ratio'] > max(B_SPY['Sharpe Ratio'], B_EW['Sharpe Ratio'])
oos_s = sens.iloc[-1]['Strategy Sharpe'] if CH.startswith('HMM') else np.nan
BUL([("The rotation improved risk-adjusted returns in the historical test, so it merits consideration as one input to the allocation process." if imp else
      "The rotation did not clearly beat the simple benchmarks on a risk-adjusted basis, so we do not recommend replacing a diversified or equity benchmark with it."),
     "Use the signal as a tilt around a core diversified portfolio rather than as an all-in switch; the 60/40 variant shows how a less concentrated version behaves.",
     (f"Treat the out-of-sample result (Sharpe {oos_s:.2f}) as the realistic expectation and the full-history result as an upper bound." if not np.isnan(oos_s) else "Treat full-history results as an upper bound on future performance."),
     f"Monitor trading costs: the strategy trades on {(turn1>0).mean():.0%} of days; the cost-adjusted result ({perf.loc[f'Regime rotation (100%, net {COST_BPS:.0f} bp)','Sharpe Ratio']:.2f} Sharpe) should be used for any implementation decision.",
     "Re-estimate periodically and review the rule after any structural change in rates, inflation or market structure."])
H('3. Factors that impact each portfolio', 2)
BUL(["Regime rotation: how long calm and stressed periods last, how quickly signals are acted on, trading costs, and the stability of the historical relationships between VIX and each asset.",
     "Equal-weight portfolio: relative performance of bonds, gold and equities; monthly rebalancing adds a small discipline benefit but offers no protection when all three fall together.",
     "Buy-and-hold SPY: equity earnings growth and valuations, interest-rate expectations and episodes of market stress, which drive the largest drawdowns.",
     "Bonds (TLT): inflation and interest-rate expectations; they hedge equity stress mainly when inflation is low and stable.",
     "Gold (GLD): real interest rates, the US dollar and safe-haven demand during geopolitical or financial stress."])

doc.add_page_break(); H('References (MLA)', 1)
for r_ in REFS: 
    p = doc.add_paragraph(r_); p.paragraph_format.left_indent = Inches(.5); p.paragraph_format.first_line_indent = Inches(-.5)
doc.save('MScFE622_GWP2_Report.docx'); print('Saved MScFE622_GWP2_Report.docx')